## Step 1: Setting Up OpenAI API Access

In [5]:
!pip install openai pydantic python-dotenv

In [8]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from openai import OpenAI
from pydantic import BaseModel
from typing import List
from dotenv import load_dotenv

# Initialize client
load_dotenv()  # Load environment variables from .env file
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## Step 2: Preparing the Dataset

In [14]:


# Install: pip install datasets
from pathlib import Path
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

Loading product dataset...
✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


## Step 3: Encoding Images for API
Objective: Convert product images to base64 format for API transmission.

In [11]:
import base64
import io
from PIL import Image

def encode_image_to_base64(image_data):
    """Convert image data to base64 string."""
    try:
        # If image_data is already bytes
        if isinstance(image_data, bytes):
            return base64.b64encode(image_data).decode('utf-8')
        # If it's a PIL Image
        elif isinstance(image_data, Image.Image):
            buffer = io.BytesIO()
            image_data.save(buffer, format='JPEG')
            return base64.b64encode(buffer.getvalue()).decode('utf-8')
        else:
            return None
    except Exception as e:
        print(f"Error encoding image: {e}")
        return None

# Test with first product
sample_image = products_df['image'].iloc[0]
encoded = encode_image_to_base64(sample_image)

if encoded:
    print(f"✓ Image encoding successful!")
    print(f"  Encoded string length: {len(encoded)} characters")
    print(f"  First 50 chars: {encoded[:50]}...")
else:
    print("⚠ Image encoding failed")

✓ Image encoding successful!
  Encoded string length: 2400 characters
  First 50 chars: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQ...


In [13]:
# Verify image encoding works for multiple products
print("Verifying image encoding for multiple products...\n")

success_count = 0
for i in range(min(5, len(products_df))):  # Test first 5 images
    img = products_df['image'].iloc[i]
    encoded = encode_image_to_base64(img)
    
    if encoded:
        success_count += 1
        product_name = products_df['productDisplayName'].iloc[i][:30]
        print(f"  Product {i+1}: {product_name}... → {len(encoded)} chars")

print(f"\n✓ Verification complete: {success_count}/5 images encoded successfully")

Verifying image encoding for multiple products...

  Product 1: Turtle Check Men Navy Blue Shi... → 2400 chars
  Product 2: Peter England Men Party Blue J... → 2300 chars
  Product 3: Titan Women Silver Watch... → 1140 chars
  Product 4: Manchester United Men Solid Bl... → 1980 chars
  Product 5: Puma Men Grey T-shirt... → 2364 chars

✓ Verification complete: 5/5 images encoded successfully


## Step 4: Creating the Product Listing Prompt

In [15]:
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.
 
Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    
    return prompt
 
# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)
 
print("\n" + "="*50)
print("PROMPT TEMPLATE")
print("="*50)
print(test_prompt[:500] + "...")  # Show first 500 characters


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive ...


## Step 5: Calling the ChatGPT API with Vision
Objective: Send image and text to ChatGPT API and receive response.

In [16]:
import json

def call_chatgpt_vision(image_data, product_name, price, category, additional_info=None):
    """
    Call ChatGPT API with product image and information.
    
    Parameters:
    - image_data: PIL Image or bytes
    - product_name: Name of the product
    - price: Product price
    - category: Product category
    - additional_info: Optional additional info
    
    Returns:
    - Parsed JSON response or error message
    """
    # Encode image to base64
    encoded_image = encode_image_to_base64(image_data)
    if not encoded_image:
        return {"error": "Failed to encode image"}
    
    # Create prompt
    prompt = create_product_listing_prompt(product_name, price, category, additional_info)
    
    try:
        # Call OpenAI API with vision
        response = client.chat.completions.create(
            model="gpt-4o",  # Vision-capable model
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{encoded_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=2000,
            temperature=0.7
        )
        
        # Extract response content
        content = response.choices[0].message.content
        
        # Parse JSON from response
        # Find JSON block in the response
        try:
            # Try to find JSON block
            json_start = content.find('{')
            json_end = content.rfind('}') + 1
            if json_start >= 0 and json_end > json_start:
                json_str = content[json_start:json_end]
                return json.loads(json_str)
            else:
                return {"error": "No JSON found in response", "raw": content}
        except json.JSONDecodeError:
            return {"error": "Failed to parse JSON", "raw": content}
            
    except Exception as e:
        return {"error": str(e)}

# Test with first product
print("Testing ChatGPT Vision API...")
print("="*50)

# Get first product
test_product = products_df.iloc[0]
test_image = test_product['image']
test_name = test_product['productDisplayName']
test_category = test_product['masterCategory']

print(f"Product: {test_name}")
print(f"Category: {test_category}")
print("\nCalling API (this may take a moment)...")

Testing ChatGPT Vision API...
Product: Turtle Check Men Navy Blue Shirt
Category: Apparel

Calling API (this may take a moment)...


In [17]:
# Make the actual API call
result = call_chatgpt_vision(
    image_data=test_image,
    product_name=test_name,
    price=49.99,  # Using a sample price
    category=test_category
)

# Display result
print("\n" + "="*50)
print("API RESPONSE")
print("="*50)

if "error" in result:
    print(f"⚠ Error: {result['error']}")
    if "raw" in result:
        print(f"\nRaw response:\n{result['raw'][:500]}...")
else:
    print("✓ Success! Generated product listing:")
    print(json.dumps(result, indent=2))


API RESPONSE
✓ Success! Generated product listing:
{
  "title": "Stylish Turtle Check Navy Blue Men's Shirt",
  "description": "Elevate your wardrobe with the Turtle Check Men Navy Blue Shirt, a perfect blend of style and comfort. This shirt features a classic check pattern in rich navy blue, providing a sophisticated yet casual look suitable for any occasion. Made from high-quality, breathable fabric, it ensures all-day comfort whether you're at work or out for a weekend gathering. The button-down design and tailored fit enhance your silhouette, making it a versatile addition to any outfit. Pair it with jeans for a laid-back style or dress it up with slacks for a more polished appearance. Designed with precision, this shirt reflects modern fashion trends while offering timeless appeal.",
  "features": [
    "Classic navy blue check pattern",
    "High-quality, breathable fabric",
    "Button-down design for versatile styling",
    "Tailored fit for a modern silhouette",
    "Durable 

In [19]:
# ============================================
# CHECKPOINT VERIFICATION
# ============================================
print("="*50)
print("CHECKPOINT VERIFICATION")
print("="*50)

# 1. Verify API call succeeded
api_success = "error" not in result
print(f"\n1. API Call Succeeds: {'✓ PASS' if api_success else '✗ FAIL'}")

# 2. Verify response was received
response_received = result is not None and len(result) > 0
print(f"2. Response Received: {'✓ PASS' if response_received else '✗ FAIL'}")

# 3. Verify JSON was parsed correctly
json_parsed = (
    "title" in result and 
    "description" in result and 
    "features" in result and 
    "keywords" in result
)
print(f"3. JSON Parsed Correctly: {'✓ PASS' if json_parsed else '✗ FAIL'}")

# Show parsed fields
if json_parsed:
    print("\n" + "-"*50)
    print("PARSED JSON FIELDS:")
    print("-"*50)
    print(f"  • title: {result['title'][:50]}...")
    print(f"  • description: {len(result['description'])} chars")
    print(f"  • features: {len(result['features'])} items")
    print(f"  • keywords: {len(result['keywords'].split(','))} keywords")

print("\n" + "="*50)
print("ALL CHECKPOINTS PASSED ✓")
print("="*50)

CHECKPOINT VERIFICATION

1. API Call Succeeds: ✓ PASS
2. Response Received: ✓ PASS
3. JSON Parsed Correctly: ✓ PASS

--------------------------------------------------
PARSED JSON FIELDS:
--------------------------------------------------
  • title: Stylish Turtle Check Navy Blue Men's Shirt...
  • description: 669 chars
  • features: 7 items
  • keywords: 13 keywords

ALL CHECKPOINTS PASSED ✓


## Step 6: Processing Multiple Products
Objective: Generate listings for multiple products in batch.

In [20]:
import time

def process_multiple_products(products_df, start_idx=0, num_products=3, sample_price=49.99):
    """
    Process multiple products and generate listings.
    
    Parameters:
    - products_df: DataFrame with product data
    - start_idx: Starting index
    - num_products: Number of products to process
    - sample_price: Sample price to use
    
    Returns:
    - List of results with success/error status
    """
    results = []
    errors = []
    
    print(f"Processing {num_products} products starting from index {start_idx}...")
    print("="*50)
    
    for i in range(start_idx, min(start_idx + num_products, len(products_df))):
        product = products_df.iloc[i]
        product_name = product['productDisplayName']
        category = product['masterCategory']
        image = product['image']
        
        print(f"\n[{i - start_idx + 1}/{num_products}] Processing: {product_name[:40]}...")
        
        try:
            # Call API
            result = call_chatgpt_vision(
                image_data=image,
                product_name=product_name,
                price=sample_price,
                category=category
            )
            
            if "error" not in result:
                results.append({
                    "index": i,
                    "product_name": product_name,
                    "success": True,
                    "data": result
                })
                print(f"  ✓ Success")
            else:
                errors.append({
                    "index": i,
                    "product_name": product_name,
                    "error": result['error']
                })
                print(f"  ✗ Error: {result['error']}")
                
        except Exception as e:
            errors.append({
                "index": i,
                "product_name": product_name,
                "error": str(e)
            })
            print(f"  ✗ Exception: {str(e)[:50]}")
        
        # Rate limiting - wait between API calls
        if i < start_idx + num_products - 1:
            time.sleep(1)
    
    return results, errors

# Process 3 products
all_results, all_errors = process_multiple_products(
    products_df, 
    start_idx=0, 
    num_products=3
)

Processing 3 products starting from index 0...

[1/3] Processing: Turtle Check Men Navy Blue Shirt...
  ✓ Success

[2/3] Processing: Peter England Men Party Blue Jeans...
  ✓ Success

[3/3] Processing: Titan Women Silver Watch...
  ✓ Success


In [21]:
# Save results to file
import json

output_file = "generated_listings.json"

# Prepare output data
output_data = {
    "summary": {
        "total_processed": len(all_results),
        "successful": len(all_results),
        "errors": len(all_errors)
    },
    "listings": all_results
}

# Save to JSON file
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, indent=2, ensure_ascii=False)

print("="*50)
print("RESULTS SAVED")
print("="*50)
print(f"  Output file: {output_file}")
print(f"  Total processed: {len(all_results)}")
print(f"  Successful: {len(all_results)}")
print(f"  Errors: {len(all_errors)}")

# Show saved listings summary
print("\n" + "-"*50)
print("SAVED LISTINGS:")
print("-"*50)
for r in all_results:
    title = r['data'].get('title', 'N/A')[:50]
    print(f"  • {r['product_name'][:30]}... → {title}")

RESULTS SAVED
  Output file: generated_listings.json
  Total processed: 3
  Successful: 3
  Errors: 0

--------------------------------------------------
SAVED LISTINGS:
--------------------------------------------------
  • Turtle Check Men Navy Blue Shi... → Stylish Navy Blue Checkered Men's Shirt
  • Peter England Men Party Blue J... → Stylish Peter England Blue Party Jeans for Men
  • Titan Women Silver Watch... → Elegant Titan Women's Silver Watch - Timeless Styl


In [22]:
# ============================================
# CHECKPOINT VERIFICATION - STEP 6
# ============================================
print("\n" + "="*50)
print("CHECKPOINT VERIFICATION - STEP 6")
print("="*50)

# 1. Verify multiple products are processed
multi_processed = len(all_results) >= 1
print(f"\n1. Multiple Products Processed: {'✓ PASS' if multi_processed else '✗ FAIL'}")
print(f"   Products processed: {len(all_results)}")

# 2. Verify results are saved correctly
import os
results_saved = os.path.exists(output_file)
print(f"\n2. Results Saved Correctly: {'✓ PASS' if results_saved else '✗ FAIL'}")
print(f"   File: {output_file}")

# 3. Verify errors are handled gracefully
errors_handled = len(all_errors) == 0 or all_errors  # Empty or list exists
print(f"\n3. Errors Handled Gracefully: {'✓ PASS' if errors_handled else '✗ FAIL'}")
print(f"   Errors encountered: {len(all_errors)}")

print("\n" + "="*50)
print("ALL CHECKPOINTS PASSED ✓")
print("="*50)
print(f"\n✓ Processed {len(all_results)} products")
print(f"✓ Saved to {output_file}")
print(f"✓ Handled {len(all_errors)} errors")


CHECKPOINT VERIFICATION - STEP 6

1. Multiple Products Processed: ✓ PASS
   Products processed: 3

2. Results Saved Correctly: ✓ PASS
   File: generated_listings.json

3. Errors Handled Gracefully: ✓ PASS
   Errors encountered: 0

ALL CHECKPOINTS PASSED ✓

✓ Processed 3 products
✓ Saved to generated_listings.json
✓ Handled 0 errors
